# 00 — M0/M1: Temeller ve I-JEPA anatomisi

Bu notebook'un amacı bir skor üretmek değil; **token akışını, stop-gradient sınırını, EMA teacher güncellemesini ve output-space masking'i** elle doğrulamaktır. Hücreleri sırayla çalıştırın. `Tahmin → Çalıştır → İncele` duraklarında cevabı önce kendiniz yazın.

Kısa akış CPU/MPS/Kaggle GPU üzerinde çalışır. Son bölümdeki resmî Meta smoke hücresi ayrı process kullanır; upstream kaynakları değiştirmez.

## Ortamı bul

Notebook'u repo içinde açın. Kaggle için önerilen yol `/kaggle/working/jepa-study`, Colab için `/content/jepa-study`'dir. Farklı bir yerdeyse `JEPA_LAB_ROOT` ortam değişkenini ayarlayabilirsiniz. İlk kurulumda terminalden `pip install -e '.[vscode]'` çalıştırın.

In [ ]:
from pathlib import Path
import os, sys

override = os.environ.get('JEPA_LAB_ROOT')
candidates = ([Path(override)] if override else []) + [
    Path.cwd(), *Path.cwd().parents,
    Path('/kaggle/working/jepa-study'), Path('/kaggle/working/I-JEPA'),
    Path('/content/jepa-study'), Path('/content/I-JEPA'),
]
ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Repo bulunamadı; JEPA_LAB_ROOT değişkenini repo yoluna ayarlayın.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('repo:', ROOT)

## Tahmin 1 — Token sayısı

Çalıştırmadan önce cevaplayın:

1. `224×224` bir görüntü `16×16` patch'lere bölünürse kaç token oluşur?
2. `16×224×224` video, `2×16×16` tubelet ile kaç `(t,h,w)` token üretir?
3. Video tensörünün laboratuvar sözleşmesi hangi eksen sırasındadır?

In [ ]:
from pprint import pprint
from jepa_lab.foundations import foundation_report

report = foundation_report()
pprint(report)
assert report['image_tokens_224_p16'] == 196
assert tuple(report['video_grid_16x224_t2_p16']) == (8, 14, 14)
assert report['video_tokens_16x224_t2_p16'] == 1568

### İncele 1

Canonical video biçimi `[B,T,C,H,W]`; Meta V-JEPA adapter'ı bunu upstream için `[B,C,T,H,W]` biçimine çevirir. `8×14×14=1568` hesabını ezberlemek yerine tubelet'in temporal stride'ını neden ikiye böldüğünü açıklayın.

## Tahmin 2 — I-JEPA maskeleri

Dört target blok birbirleriyle örtüşebilir. Fakat context ile target neden ayrık olmalıdır? Görselde `0=context`, `1..4=target block`, `-1=kullanılmayan patch` olacaktır. Önce haritanın yaklaşık nasıl görüneceğini çizin.

In [ ]:
%matplotlib inline
import torch
from jepa_lab.device import seed_everything, select_device
from jepa_lab.masking import MultiBlockMasker
from jepa_lab.visualization import plot_image_masks

seed_everything(42)
device = select_device('auto')
masker = MultiBlockMasker(
    (8, 8), num_targets=4, target_scale=(0.15, 0.20), context_scale=(0.85, 1.0)
)
masks = masker(2, generator=torch.Generator().manual_seed(42), device=device)
display(plot_image_masks(masks, sample=0))
context_set = set(masks.context[0].cpu().tolist())
target_sets = [set(block.cpu().tolist()) for block in masks.targets[0]]
print('context shape:', tuple(masks.context.shape))
print('targets shape:', tuple(masks.targets.shape))
print('context-target disjoint:', all(context_set.isdisjoint(t) for t in target_sets))
print('target-target overlap exists:', any(target_sets[i] & target_sets[j] for i in range(4) for j in range(i+1, 4)))

## Tahmin 3 — Forward akışı

Aşağıdaki model `64/8=8` patch grid kullanır. Teacher'ın gördüğü token sayısı ile context encoder'ın gördüğü token sayısını tahmin edin. Target seçimi görüntü üzerinde mi, teacher encoder çıktısında mı yapılmalıdır?

In [ ]:
from jepa_lab.image_jepa import ImageJEPA

model = ImageJEPA(
    image_size=64, patch_size=8, embed_dim=64,
    encoder_depth=2, encoder_heads=4,
    predictor_dim=64, predictor_depth=2, predictor_heads=4,
).to(device)
images = torch.randn(2, 3, 64, 64, device=device)
with torch.no_grad():
    full_teacher = model.target_encoder(images)
output = model(images, masks)
print('full teacher [B,N,D]:', tuple(full_teacher.shape))
print('context [B,Nc,D]:', tuple(output.context.shape))
print('prediction [B,K,Nt,D]:', tuple(output.predicted.shape))
print('selected target:', tuple(output.target.shape))
print('SmoothL1:', float(output.loss.detach()))
assert full_teacher.shape[1] == 64
assert output.predicted.shape == output.target.shape

### İncele 3

Kritik ayrım: target encoder **tam görüntüyü** kodlar; hedef patch'ler latent çıktıdan seçilir. Resmî kod, paper'daki yalın squared-L2 anlatımından farklı olarak target'a parametresiz LayerNorm ve `SmoothL1` uygular.

## Tahmin 4 — Gradient ve EMA

Backward sonrasında hangi modüllerde gradient bekliyorsunuz: context encoder, predictor, target encoder? Momentum `m=0.996` iken target parametresinin yeni değeri hangi iki parametrenin ağırlıklı toplamıdır?

In [ ]:
from dataclasses import asdict
from jepa_lab.image_jepa import train_one_step

optimizer = torch.optim.AdamW(model.trainable_parameters(), lr=1e-3)
step_result = train_one_step(model, images, masks, optimizer, ema_momentum=0.996)
pprint(asdict(step_result))
assert step_result.context_grad_norm > 0
assert step_result.predictor_grad_norm > 0
assert not step_result.target_has_grad
assert step_result.all_finite
assert step_result.ema_max_error < 1e-6

## Resmî Meta I-JEPA smoke — ayrı process

Bu hücre büyük checkpoint indirmez; pinned `facebookresearch/ijepa` bileşenleriyle tek bir gerçek forward/backward adımı çalıştırır. CPU'da da çalışır fakat birkaç dakika sürebilir. Upstream `src` namespace çakışmasını önlemek için subprocess zorunludur. `RUN_OFFICIAL=True` yaparak bilinçli biçimde başlatın.

In [ ]:
import subprocess

RUN_OFFICIAL = False
if RUN_OFFICIAL:
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_ijepa_smoke.py'),
        '--device', str(device), '--image-size', '112', '--batch-size', '1',
    ], cwd=ROOT, check=True)
else:
    print('Atlandı. Hazır olduğunuzda RUN_OFFICIAL=True yapın.')

## Resmî pretrained I-JEPA encoder feature'ları

Checkpoint'i `configs/checkpoints.yaml` içindeki resmî URL'den indirin. ViT-H/14 büyük olduğu için bu hücre Kaggle GPU içindir. Script patch latent'lerini, mean-pooling cosine değerlerini, checkpoint SHA-256 ve missing/unexpected key listesini kaydeder. Random-init shape smoke ile pretrained sonucu aynı şey değildir.

In [ ]:
RUN_PRETRAINED_IJEPA = False
ijepa_checkpoint = ROOT / 'checkpoints/IN1K-vit.h.14-300e.pth.tar'
if RUN_PRETRAINED_IJEPA:
    if not ijepa_checkpoint.is_file():
        raise FileNotFoundError("Resmî I-JEPA checkpoint'ini önce checkpoints/ altına indirin.")
    subprocess.run([
        sys.executable, str(ROOT / 'scripts/official_ijepa_features.py'),
        '--device', str(device), '--model', 'vit_huge', '--patch-size', '14',
        '--image-size', '224', '--checkpoint', str(ijepa_checkpoint),
        '--output', str(ROOT / 'runs/ijepa_pretrained_views.npz'),
    ], cwd=ROOT, check=True)
else:
    print('Atlandı. Checkpoint ve Kaggle GPU hazır olduğunda bayrağı açın.')

## M0/M1 geçiş kontrolü

Bir sonraki notebook'a geçmeden önce şu cümleleri şekillerle açıklayabilmelisiniz:

- `224/16 → 14×14=196`; video için `16/2×14×14=1568`.
- Context encoder yalnız görünen patch'leri, EMA target encoder tam görüntüyü görür.
- Target seçimi encoder çıkışında yapılır; target gradient almaz.
- Predictor konum bilgisini mask token/position embedding üzerinden alır.
- EMA bir optimizer adımı değildir; online parametreleri yavaşça target'a taşır.